# Neural CFR+ branching ladder

This notebook separates two questions:

1. What frontier and saved-record growth should full-expansion external sampling produce under the initial uniform policy?
2. What batch size actually fits and runs efficiently on the current GPU?

Every live profile runs in a fresh subprocess. An OOM therefore kills only that measurement, not the notebook kernel. The profiler performs production forward expansion, reverse backups, and target construction, but does not commit records into replay.

In [ ]:
from pathlib import Path
import json
import math
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.eval.profile_cfr_plus_branching import analytical_branching_profile
from liars_poker.env import rules_for_spec

assert torch.cuda.is_available(), 'The live ladder is intended for a CUDA machine.'
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
free_bytes, total_bytes = torch.cuda.mem_get_info()
print('free / total GPU GiB:', (free_bytes / 2**30, total_bytes / 2**30))

In [ ]:
SPECS = {
    '30 claims: r5, no FullHouse': GameSpec(
        ranks=5, suits=4, hand_size=3,
        claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'Quads'),
        suit_symmetry=True,
    ),
    '39 claims: r6, no FullHouse': GameSpec(
        ranks=6, suits=4, hand_size=3,
        claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'Quads'),
        suit_symmetry=True,
    ),
    '50 claims: r5, with FullHouse': GameSpec(
        ranks=5, suits=4, hand_size=3,
        claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
        suit_symmetry=True,
    ),
    '69 claims: r6, with FullHouse': GameSpec(
        ranks=6, suits=4, hand_size=4,
        claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
        suit_symmetry=True,
    ),
}

{label: len(rules_for_spec(spec).claims) for label, spec in SPECS.items()}

## Analytical estimate before allocating GPU frontiers

At initialization, zero regrets produce uniform play. Traverser nodes expand every higher claim; opponent nodes sample one action from `CALL + higher claims`. This recurrence gives expected active rows, edges, and saved records per initial deal without constructing the tree.

In [ ]:
analytical = {}
summary_rows = []
depth_rows = []
for label, spec in SPECS.items():
    for traverser in (0, 1):
        result = analytical_branching_profile(spec, traverser=traverser)
        analytical[(label, traverser)] = result
        summary_rows.append({
            'spec': label,
            'traverser': traverser,
            'claims': result['claim_count'],
            'history words': result['history_word_count'],
            'history bytes / row': result['history_bytes_per_row'],
            'peak rows / deal': result['peak_rows_per_deal'],
            'traverser edges / deal': result['total_traverser_edges_per_deal'],
            'opponent continuations / deal': result['total_opponent_continuations_per_deal'],
            'records / deal': result['total_records_per_deal'],
        })
        for row in result['depth_profile']:
            depth_rows.append({'spec': label, 'traverser': traverser, **row})

analytical_summary = pd.DataFrame(summary_rows)
analytical_depth = pd.DataFrame(depth_rows)
analytical_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for (label, traverser), group in analytical_depth.groupby(['spec', 'traverser']):
    short = label.split(':')[0]
    axes[0].plot(group['depth'], group['active_rows_in_per_deal'], label=f'{short}, P{traverser + 1}')
    axes[1].plot(group['depth'], group['traverser_claim_edges_per_deal'], label=f'{short}, P{traverser + 1}')
axes[0].set_yscale('log')
axes[1].set_yscale('log')
axes[0].set(title='Expected active frontier per initial deal', xlabel='depth', ylabel='rows')
axes[1].set(title='Expected traverser claim edges per initial deal', xlabel='depth', ylabel='edges')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()

## Conservative initial batch-size guesses

The 30-claim game at batch 1024 is the anchor. One estimate scales by peak frontier rows; another scales by total saved records. Their minimum is the initial working estimate. This is not a memory proof—network activations and allocator behavior still require live measurement.

In [ ]:
ANCHOR_BATCH = 1024
anchor_label = next(label for label in SPECS if label.startswith('30 claims'))
anchor_peak = max(analytical[(anchor_label, p)]['peak_rows_per_deal'] for p in (0, 1))
anchor_records = max(analytical[(anchor_label, p)]['total_records_per_deal'] for p in (0, 1))

batch_rows = []
for label in SPECS:
    peak = max(analytical[(label, p)]['peak_rows_per_deal'] for p in (0, 1))
    records = max(analytical[(label, p)]['total_records_per_deal'] for p in (0, 1))
    row_limited = max(1, math.floor(ANCHOR_BATCH * anchor_peak / peak))
    record_limited = max(1, math.floor(ANCHOR_BATCH * anchor_records / records))
    conservative = min(row_limited, record_limited)
    conservative_power_two = 2 ** math.floor(math.log2(conservative))
    batch_rows.append({
        'spec': label,
        'row-limited batch': row_limited,
        'record-limited batch': record_limited,
        'initial power-of-two estimate': conservative_power_two,
    })
batch_estimates = pd.DataFrame(batch_rows).set_index('spec')
batch_estimates

## Isolated live profiler

Each call starts a new Python process. The adaptive screen begins below the analytical estimate and doubles until it hits an error, timeout, or 80% of total VRAM.

In [ ]:
PROFILE_REGRET_HIDDEN_SIZES = (2048, 2048)

def profile_subprocess(label, spec, traverser, batch_size, timeout_s=300):
    command = [
        sys.executable, '-m', 'liars_poker.eval.profile_cfr_plus_branching',
        '--ranks', str(spec.ranks),
        '--suits', str(spec.suits),
        '--hand-size', str(spec.hand_size),
        '--claim-kinds', *spec.claim_kinds,
        '--traverser', str(traverser),
        '--batch-size', str(batch_size),
        '--device', 'cuda',
        '--hidden-sizes', *map(str, PROFILE_REGRET_HIDDEN_SIZES),
    ]
    if spec.suit_symmetry:
        command.append('--suit-symmetry')
    started = time.perf_counter()
    try:
        completed = subprocess.run(
            command,
            cwd=REPO_ROOT,
            capture_output=True,
            text=True,
            timeout=timeout_s,
        )
        lines = [line for line in completed.stdout.splitlines() if line.strip()]
        result = json.loads(lines[-1]) if lines else {
            'status': 'error',
            'error': completed.stderr[-2000:],
        }
        result['returncode'] = completed.returncode
    except subprocess.TimeoutExpired:
        result = {'status': 'timeout', 'error': f'exceeded {timeout_s}s'}
    result.update({
        'label': label,
        'traverser': traverser,
        'batch_size': batch_size,
        'parent_wall_s': time.perf_counter() - started,
    })
    return result

def adaptive_batches(initial_estimate):
    start = max(1, 2 ** math.floor(math.log2(max(1, initial_estimate // 4))))
    return [start * (2 ** power) for power in range(7)]

In [ ]:
MAX_VRAM_FRACTION = 0.80
PROFILE_TIMEOUT_S = 300
live_results = []

for label, spec in SPECS.items():
    estimate = int(batch_estimates.loc[label, 'initial power-of-two estimate'])
    for traverser in (0, 1):
        for batch_size in adaptive_batches(estimate):
            result = profile_subprocess(
                label,
                spec,
                traverser,
                batch_size,
                timeout_s=PROFILE_TIMEOUT_S,
            )
            live_results.append(result)
            peak_gib = result.get('peak_reserved_bytes', 0) / 2**30
            print(label, 'P' + str(traverser + 1), 'batch', batch_size, result['status'], f'peak={peak_gib:.2f} GiB')
            if result['status'] != 'complete':
                break
            if result.get('peak_reserved_bytes', 0) >= MAX_VRAM_FRACTION * total_bytes:
                break

In [ ]:
live_summary = pd.DataFrame([
    {
        'spec': result['label'],
        'traverser': result['traverser'],
        'batch size': result['batch_size'],
        'status': result['status'],
        'peak rows': result.get('peak_rows'),
        'regret records': result.get('regret_records'),
        'strategy records': result.get('strategy_records'),
        'setup s': result.get('setup_s'),
        'traversal s': result.get('traversal_s'),
        'total child wall s': result.get('wall_s'),
        'peak allocated GiB': result.get('peak_allocated_bytes', 0) / 2**30,
        'peak reserved GiB': result.get('peak_reserved_bytes', 0) / 2**30,
        'error': result.get('error'),
    }
    for result in live_results
])
live_summary

In [ ]:
complete = live_summary[live_summary['status'] == 'complete']
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for (label, traverser), group in complete.groupby(['spec', 'traverser']):
    short = label.split(':')[0]
    legend = f'{short}, P{traverser + 1}'
    group = group.sort_values('batch size')
    axes[0].plot(group['batch size'], group['peak reserved GiB'], marker='o', label=legend)
    axes[1].plot(group['batch size'], group['traversal s'], marker='o', label=legend)
    axes[2].plot(group['batch size'], group['peak rows'], marker='o', label=legend)
for ax in axes:
    ax.set_xscale('log', base=2)
    ax.grid(alpha=0.25)
axes[0].set(title='GPU memory by batch', xlabel='initial deals', ylabel='reserved GiB')
axes[1].set(title='Traversal profile wall time', xlabel='initial deals', ylabel='seconds')
axes[2].set(title='Peak live frontier', xlabel='initial deals', ylabel='rows')
axes[2].set_yscale('log')
axes[0].legend(fontsize=8)
plt.tight_layout()

## Per-depth inspection

Choose any completed point to inspect where rows, edges, time, and memory peak. This is the information needed to decide whether the next intervention should be smaller batches, traverser-action sampling, or a different traversal representation.

In [ ]:
chosen = next(result for result in reversed(live_results) if result['status'] == 'complete')
chosen_depth = pd.DataFrame(chosen['depth_profile'])
chosen_depth['allocated GiB'] = chosen_depth['allocated_bytes'] / 2**30
chosen_depth['reserved GiB'] = chosen_depth['reserved_bytes'] / 2**30
chosen_depth

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.ravel()
axes[0].plot(chosen_depth['depth'], chosen_depth['active_rows_in'], marker='o', label='active rows')
axes[0].plot(chosen_depth['depth'], chosen_depth['active_rows_out'], marker='o', label='next rows')
axes[0].set_yscale('log')
axes[0].legend()
axes[1].plot(chosen_depth['depth'], chosen_depth['traverser_claim_edges_expanded'], marker='o', label='traverser edges')
axes[1].plot(chosen_depth['depth'], chosen_depth['opponent_continuations'], marker='o', label='opponent continuations')
axes[1].set_yscale('symlog', linthresh=1)
axes[1].legend()
axes[2].plot(chosen_depth['depth'], chosen_depth['forward_s'], marker='o', label='forward')
axes[2].plot(chosen_depth['depth'], chosen_depth['backup_s'], marker='o', label='backup')
axes[2].legend()
axes[3].plot(chosen_depth['depth'], chosen_depth['allocated GiB'], marker='o', label='allocated')
axes[3].plot(chosen_depth['depth'], chosen_depth['reserved GiB'], marker='o', label='reserved')
axes[3].legend()
for ax, title in zip(axes, ('Frontier rows', 'Branching', 'Time per depth', 'GPU memory')):
    ax.set(title=title, xlabel='depth')
    ax.grid(alpha=0.25)
plt.tight_layout()